# Notebook 10 — Train RoadFlood-VLM

Set `ROADFLOOD_RUN_MODE=smoke` for a 2-step test or `ROADFLOOD_RUN_MODE=full` for complete training.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, PeftModel
from qwen_vl_utils import process_vision_info
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
    set_seed,
)


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Run from the ResilientVLM repository or a subdirectory.")


PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "data" / "processed" / "vlm_dataset"
SPLIT_DIR = PROJECT_ROOT / "data" / "processed" / "sturm_vlm_quick" / "training_splits"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "sturm_auxiliary_training"
RUN_MODE = os.environ.get("ROADFLOOD_RUN_MODE", "smoke").lower().strip()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("ROADFLOOD_RUN_MODE must be smoke or full")

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = OUTPUT_ROOT / f"{RUN_MODE}_{STAMP}"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ADAPTER_DIR = RUN_DIR / "final_adapter"
PREDICTION_DIR = RUN_DIR / "predictions"
for path in [RUN_DIR, CHECKPOINT_DIR, ADAPTER_DIR, PREDICTION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

MODEL_ID = os.environ.get("ROADFLOOD_MODEL_ID", "").strip()
if not MODEL_ID:
    config_candidates = [
        PROJECT_ROOT / "outputs" / "vlm_baselines" / "experiment_configuration.json",
        PROJECT_ROOT / "outputs" / "vlm_baselines" / "baseline_experiment_configuration.json",
    ]
    for config_path in config_candidates:
        if not config_path.exists():
            continue
        try:
            config = json.loads(config_path.read_text(encoding="utf-8"))
            MODEL_ID = (
                config.get("model_id")
                or config.get("model", {}).get("model_id")
                or config.get("model", {}).get("id")
                or ""
            )
            if MODEL_ID:
                break
        except Exception:
            pass
if not MODEL_ID:
    MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"


def first_existing_path(candidates: list[Path], label: str) -> Path:
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {label}: {candidates}")


TRAIN_CSV = first_existing_path([SPLIT_DIR / "train.csv", SPLIT_DIR / "training.csv"], "train split")
VAL_CSV = first_existing_path([SPLIT_DIR / "validation.csv", SPLIT_DIR / "val.csv", SPLIT_DIR / "valid.csv"], "validation split")
TEST_CSV = first_existing_path([SPLIT_DIR / "test.csv"], "test split")

TEXT_COLUMNS = ["multimodal_prompt", "instruction_text", "instruction", "input"]
RESPONSE_COLUMNS = ["response_text", "response"]
S2_COLUMNS = ["training_s2_path", "s2_png_path", "training_s2_relative_path", "s2_png_relative_path"]
S1_COLUMNS = ["training_s1_path", "s1_png_path", "training_s1_relative_path", "s1_png_relative_path"]


def first_existing_column(df: pd.DataFrame, candidates: list[str], label: str) -> str:
    for column in candidates:
        if column in df.columns:
            return column
    raise KeyError(f"No {label} column found. Expected one of {candidates}")


def resolve_path(value: Any) -> Path:
    raw = "" if pd.isna(value) else str(value).strip()
    if not raw:
        return Path("")
    direct = Path(raw)
    if direct.is_absolute() and direct.exists():
        return direct
    normalized = raw.replace("\\", "/")
    for marker in ["data/", "training_images/"]:
        index = normalized.find(marker)
        if index >= 0:
            suffix = Path(normalized[index:])
            for root in [PROJECT_ROOT, DATASET_ROOT]:
                candidate = root / suffix
                if candidate.exists():
                    return candidate.resolve()
    for root in [PROJECT_ROOT, DATASET_ROOT, SPLIT_DIR]:
        candidate = root / direct
        if candidate.exists():
            return candidate.resolve()
    return direct


def prepare_frame(path: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    text_col = first_existing_column(df, TEXT_COLUMNS, "prompt")
    response_col = first_existing_column(df, RESPONSE_COLUMNS, "response")
    s2_col = first_existing_column(df, S2_COLUMNS, "Sentinel-2")
    s1_col = first_existing_column(df, S1_COLUMNS, "Sentinel-1")
    df["prompt"] = df[text_col].fillna("").astype(str).str.strip()
    df["answer"] = df[response_col].fillna("").astype(str).str.strip()
    df["s2"] = df[s2_col].map(resolve_path)
    df["s1"] = df[s1_col].map(resolve_path)
    valid = df["prompt"].ne("") & df["answer"].ne("") & df["s2"].map(Path.exists) & df["s1"].map(Path.exists)
    invalid = df.loc[~valid]
    if not invalid.empty:
        invalid.to_csv(RUN_DIR / f"{split_name}_invalid.csv", index=False)
    df = df.loc[valid].reset_index(drop=True)
    if df.empty:
        raise RuntimeError(f"No valid records remain in {split_name}")
    return df


train_df = prepare_frame(TRAIN_CSV, "train")
val_df = prepare_frame(VAL_CSV, "validation")
test_df = prepare_frame(TEST_CSV, "test")
if RUN_MODE == "smoke":
    train_df = train_df.head(min(8, len(train_df))).copy()
    val_df = val_df.head(min(4, len(val_df))).copy()
    test_df = test_df.head(min(4, len(test_df))).copy()

SYSTEM_PROMPT = (
    "You are RoadFlood-VLM, a transportation-flood assessment assistant. "
    "Use the Sentinel-2 optical image, Sentinel-1 radar image, and supplied "
    "transportation context. Produce a concise, evidence-grounded response."
)


class RoadFloodDataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, index: int) -> dict[str, str]:
        row = self.frame.iloc[index]
        return {
            "instruction_id": str(row.get("instruction_id", f"record_{index}")),
            "scene_id": str(row.get("scene_id", "")),
            "prompt": row["prompt"],
            "answer": row["answer"],
            "s2": str(row["s2"]),
            "s1": str(row["s1"]),
        }


train_dataset = RoadFloodDataset(train_df)
val_dataset = RoadFloodDataset(val_df)
test_dataset = RoadFloodDataset(test_df)

use_4bit = os.environ.get("ROADFLOOD_USE_4BIT", "1") == "1" and torch.cuda.is_available()
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16 if torch.cuda.is_available() else torch.float32
quantization_config = None
if use_4bit:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
    )

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": dtype,
    "low_cpu_mem_usage": True,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"
if quantization_config is not None:
    model_kwargs["quantization_config"] = quantization_config

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **model_kwargs)
model.config.use_cache = False
if use_4bit:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

EXISTING_ADAPTER_DIR = Path(
    os.environ.get(
        "ROADFLOOD_EXISTING_ADAPTER",
        str(
            PROJECT_ROOT
            / "outputs"
            / "roadflood_vlm_training"
            / "full_20260729T032411Z"
            / "final_adapter"
        ),
    )
)
if EXISTING_ADAPTER_DIR.exists():
    print(f"Loading existing adapter: {EXISTING_ADAPTER_DIR}")
    model=PeftModel.from_pretrained(
        model,
        EXISTING_ADAPTER_DIR,
        is_trainable=True,
    )
else:
    print("Existing adapter not found. Creating new LoRA.")
    model=get_peft_model(
        model,
        lora_config,
    )

model.print_trainable_parameters()


def build_messages(example: dict[str, str], include_answer: bool) -> list[dict[str, Any]]:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": example["s2"]},
                {"type": "image", "image": example["s1"]},
                {"type": "text", "text": example["prompt"]},
            ],
        },
    ]
    if include_answer:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": example["answer"]}]})
    return messages


class RoadFloodCollator:
    def __init__(self, processor: Any):
        self.processor = processor

    def _process(self, message_batch: list[list[dict[str, Any]]], add_generation_prompt: bool) -> dict[str, torch.Tensor]:
        texts = [
            self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
            )
            for messages in message_batch
        ]
        image_inputs = []
        video_inputs = []
        for messages in message_batch:
            images, videos = process_vision_info(messages)
            image_inputs.append(images)
            if videos:
                video_inputs.append(videos)
        processor_kwargs = {
            "text": texts,
            "images": image_inputs,
            "padding": True,
            "return_tensors": "pt",
        }
        if video_inputs:
            processor_kwargs["videos"] = video_inputs
        return self.processor(**processor_kwargs)

    def __call__(self, examples: list[dict[str, str]]) -> dict[str, torch.Tensor]:
        full_messages = [build_messages(example, True) for example in examples]
        prompt_messages = [build_messages(example, False) for example in examples]
        full_batch = self._process(full_messages, False)
        prompt_batch = self._process(prompt_messages, True)
        labels = full_batch["input_ids"].clone()
        labels[full_batch["attention_mask"] == 0] = -100
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1).tolist()
        for row, length in enumerate(prompt_lengths):
            labels[row, : int(length)] = -100
        for token in ["<|image_pad|>", "<|video_pad|>", "<|vision_start|>", "<|vision_end|>"]:
            token_id = self.processor.tokenizer.convert_tokens_to_ids(token)
            if token_id is not None and token_id != self.processor.tokenizer.unk_token_id:
                labels[labels == token_id] = -100
        full_batch["labels"] = labels
        return full_batch


collator = RoadFloodCollator(processor)
probe = collator([train_dataset[0]])
if int((probe["labels"] != -100).sum()) == 0:
    raise RuntimeError("Collator produced no supervised assistant tokens")

if RUN_MODE == "smoke":
    epochs, max_steps, grad_accum, eval_steps, save_steps = 1.0, 2, 1, 1, 1
else:
    epochs = float(os.environ.get("ROADFLOOD_EPOCHS", "3"))
    max_steps = -1
    grad_accum = int(os.environ.get("ROADFLOOD_GRAD_ACCUM", "8"))
    eval_steps = int(os.environ.get("ROADFLOOD_EVAL_STEPS", "25"))
    save_steps = int(os.environ.get("ROADFLOOD_SAVE_STEPS", "25"))

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=epochs,
    max_steps=max_steps,
    per_device_train_batch_size=int(os.environ.get("ROADFLOOD_TRAIN_BATCH", "1")),
    per_device_eval_batch_size=int(os.environ.get("ROADFLOOD_EVAL_BATCH", "1")),
    gradient_accumulation_steps=grad_accum,
    learning_rate=float(os.environ.get("ROADFLOOD_LR", "2e-4")),
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    optim="paged_adamw_8bit" if use_4bit else "adamw_torch",
    logging_strategy="steps",
    logging_steps=1 if RUN_MODE == "smoke" else 5,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    processing_class=processor,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

result = trainer.train()
trainer.save_model(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))
pd.DataFrame(trainer.state.log_history).to_csv(RUN_DIR / "trainer_log_history.csv", index=False)

summary = {
    "run_mode": RUN_MODE,
    "model_id": MODEL_ID,
    "use_4bit": use_4bit,
    "dtype": str(dtype),
    "train_records": len(train_dataset),
    "validation_records": len(val_dataset),
    "test_records": len(test_dataset),
    "metrics": result.metrics,
    "adapter_dir": str(ADAPTER_DIR),
}
(RUN_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
